# DCAT-AP-NL Requirement Analysis for Dataverse Metadata Exporter



In [23]:
# boiler plate functions
# imports SPARQL prefixes and functions defs

from pprint import pprint
# from SPARQLWrapper import SPARQLWrapper, JSON, TURTLE, CSV 
from rdflib import Graph

prefixes = '''    
PREFIX adms: <http://www.w3.org/ns/adms#>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcatap: <http://data.europa.eu/r5r/>
PREFIX eli: <http://data.europa.eu/eli/ontology#>
PREFIX eush: <https://purl.eu/ns/shacl#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX vcard: <http://www.w3.org/2006/vcard/ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

PREFIX dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/>
'''    

def sparql_file_query(query, format, filepath):
    g = Graph()
    g.parse(filepath, format=format)  # can also use "ttl" for Turtle
    query = prefixes + query
    results = g.query(query)
    return results

# def sparql_query(query, format):
#     formats = {"json": JSON, "turtle": TURTLE, "csv": CSV}
#     f_ = formats[format]
#     endpoint = "http://vocab.getty.edu/sparql"
#     sparql = SPARQLWrapper(endpoint)
#     query = prefixes + query     
#     sparql.setQuery(query)
#     sparql.setReturnFormat(f_)
#     results = sparql.query().convert()    
#     return results


# def print_sparql_results(results):
#     for row in results["results"]["bindings"]:
#         return (row)


In [24]:
# in this query I am looking at the pattern of the shacl shapes for 1 mandatory property, namely
# dcat:Dataset: dct:description

sparql_dcatap_dataset_1mandatory_props = ''' 
DESCRIBE ?prop_shape
WHERE {
    BIND(dct:description AS ?prop_path) .
    <https://semiceu.github.io//DCAT-AP/releases/3.0.1#DatasetShape> sh:property ?prop_shape .
    ?prop_shape sh:path ?prop_path .
}
'''
results = sparql_file_query(query=sparql_dcatap_dataset_1mandatory_props, 
                            format='ttl', 
                            filepath='dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl')
print(results.serialize(format='ttl').decode('utf-8'))


# Requirement: Mandatory Dataset properties - Research

**What the mandatory properties of dcat:Dataset in DCAT-AP?**
ie. `dct:description`

**What the mandatory properties of dcat:Dataset in DCAT-AP-NL?**
ie. `dct:identifier`

* include cardinality and property range


According to https://docs.geostandaarden.nl/dcat/dcat-ap-nl30/#17C1E0BE
> The SHACL rules of DCAT-AP-NL build on the SHACL rules from DCAT API-3.0. **All the rules from DCAT-AP-3.0 (see [DCAT AP-3.0 dcat-ap-SHACL.ttl](dcat-ap/releases/3.0.0/shacl/dcat-ap-SHACL.ttl)) are still applicable. DCAT-AP-NL only tightens some data rules.**

>To test whether a dataset description meets DCAT-AP-NL, it is also necessary to include both the DCAT-AP SHACL shapes and the DCAT-AP-NL SHACL shapes in the validation.

> To properly support the validation of the dataset descriptions, DCAT-AP breaks the SHACL shapes is split to support different validation scenarios and aspects. See the chapter **[Validation of DCAT-AP](https://semiceu.github.io/DCAT-AP/releases/3.0.0/#validation-of-dcat-ap)**.

**DCAT-AP-NL SHACL shapes** are divided into:

* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl](dcat-ap-nl-SHACL.ttl): The SHACL shapes of DCAT-AP-NL, excluding the validation rules around the class range of properties.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik.ttl](dcat-ap-nl-SHACL-klassebereik.ttl) The SHACL shapes of DCAT-AP-EN for validating the class range of properties, excluding the class range of properties with a value derived from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl](dcat-ap-nl-SHACL-klassebereik-codelijsten.ttl:) dcat-ap-nl-SHACL class range-codelists.ttl : The SHACL shapes of DCAT-AP-EN for validating the class range of properties with a value from a code list.
* [dcat-ap-nl/shapes/dcat-ap-nl-SHACL-aanbevolen.ttl](dcat-ap-nl-SHACL-aanbevolen.ttl): The SHACL shapes of DCAT-AP-EN for validating recommended properties.


## Note on DCAT-AP versions
<div class="alert alert-block alert-warning">
Note that while, at moment of writing DCAT-AP-NL is basing is profile in DCAT-AP v3.0.0, from June 2023.</br>
DCAT-AP v3.0.1 includes some important changes (see <a href="https://github.com/SEMICeu/DCAT-AP/blob/master/releases/3.0.1/CHANGELOG.md">3.0.1/CHANGELOG.md</a>) including the dcat-ap/releases/3.0.1/shacl/ranges.ttl which is important for these requirements.
Hence this requirement analysis will be based on DCAT-AP 3.0.1, although DCAT-AP-NL, at the time of writing (2026.02.10) is still based on DCAT-AP 3.0.0  
</div>


The pattern I see in the cell above (`dcat:Dataset: dct:description`) property shapes, makes me conclude that 
* in **DCAT-AP SHACL the required properties have `shacl:minCount 1`**, which makes sense

Follow-up questions/queries?

* which other Dataset properties have `shacl:minCount 1` AKA are mandatory? 
* is the same pattern present in DCAT-AP-NL shacl?

In [38]:
# which Dataset properties have `shacl:minCount 1` AKA are mandatory DCAT-AP? 

sparql_dcatap_dataset_mandatory_props = ''' 
DESCRIBE ?prop_shape
WHERE {
    
<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape> sh:property ?prop_shape .
    ?prop_shape sh:minCount 1 ;
                sh:path ?prop_path .
}

'''
results = sparql_file_query(query=sparql_dcatap_dataset_mandatory_props, 
                            format='ttl', 
                            filepath='dcat-ap/releases/3.0.1/shacl/dcat-ap-SHACL.ttl')
print(results.serialize(format='ttl').decode('utf-8'))



@prefix dc1: <http://purl.org/dc/terms/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix shacl: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/8f61614144aa7bca188b24f5976593dc08aad0e6> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.description" ;
    shacl:description "A free-text account of the Dataset."@en ;
    shacl:minCount 1 ;
    shacl:name "description"@en ;
    shacl:path dc1:description .

<https://semiceu.github.io/DCAT-AP/releases/3.0.1#dcat:DatasetShape/f8b02efd063b8089b72ce9677a1e4a3488eeb9a9> rdfs:seeAlso "https://semiceu.github.io/DCAT-AP/releases/3.0.1#Dataset.title" ;
    shacl:description "A name given to the Dataset."@en ;
    shacl:minCount 1 ;
    shacl:name "title"@en ;
    shacl:path dc1:title .




In [26]:
# which Dataset properties have `shacl:minCount 1` AKA are mandatory in DCAT-AP-NL? 

sparql_dcatap_dataset_mandatory_props = ''' 
DESCRIBE ?prop_shape
WHERE {
    dcatapnl-sh:DatasetShape sh:property ?prop_shape .
    ?prop_shape sh:minCount 1 ;
                sh:path ?prop_path .
}

'''
results = sparql_file_query(query=sparql_dcatap_dataset_mandatory_props, 
                            format='ttl', 
                            filepath='dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl')
print(results.serialize(format='ttl').decode('utf-8'))
# for row in results:
#     print(row)

@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dcatapnl-sh: <http://modellen.geostandaarden.nl/dcat-ap-nl/id/shape/> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix eush: <https://purl.eu/ns/shacl#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

dcatapnl-sh:DatasetShape_accessRights_minCount sh:minCount 1 ;
    sh:name "access rights"@en ;
    sh:path dct:accessRights ;
    eush:message "Minimally 1 values are expected for access rights"@en .

dcatapnl-sh:DatasetShape_contactPoint_minCount sh:minCount 1 ;
    sh:name "contact point"@en ;
    sh:path dcat:contactPoint ;
    eush:message "Minimally 1 values are expected for contact point"@en .

dcatapnl-sh:DatasetShape_creator_minCount sh:minCount 1 ;
    sh:name "creator"@en ;
    sh:path dct:creator ;
    eush:message "Minimally 1 values are expected for creator"@en .

dcatapnl-sh:DatasetShape_identifier_minCount sh:minCount 1 ;
    sh:name "identifier"@en ;
    sh:path dct:

# Requirement: Mandatory Dataset properties - Outcome

The response to the query above, tells me that in addition to 

**DCAT-AP mandatory properties of dcat:Dataset:**

*  http://purl.org/dc/terms/description
*  http://purl.org/dc/terms/title 

**DCAT-AP-NL mandatory properties of dcat:Dataset:**

* http://purl.org/dc/terms/accessRights 
* http://www.w3.org/ns/dcat#contactPoint 
* http://purl.org/dc/terms/creator 
* http://purl.org/dc/terms/identifier 
* http://purl.org/dc/terms/publisher 
* http://www.w3.org/ns/dcat#theme


In [27]:
sparql_mandatory_dataset_propos = '''
SELECT ?targetClass ?prop_path ?nodeKind
WHERE {
    BIND( dcat:Dataset AS ?targetClass )
    ?shape a sh:NodeShape ;
           sh:targetClass ?targetClass ;
           sh:property ?prop_shape .
    ?prop_shape sh:minCount 1 ; # mandatory prop 
                sh:path ?prop_path .
    OPTIONAL { ?prop_shape sh:nodeKind ?nodeKind }
    
}
'''
# sparql_mandatory_dataset_propos = '''
# SELECT *
# WHERE {
#     ?shape a sh:NodeShape ;
#            sh:targetClass dcat:Dataset;
#            sh:property ?prop_shape .
#     ?prop_shape sh:minCount 1 ; # mandatory prop 
#                 sh:path ?prop_path
# }
# '''

results = sparql_file_query(query=sparql_mandatory_dataset_propos, 
                            format='ttl', 
                            filepath='dcat-ap-nl/shapes/dcat-ap-nl-SHACL.ttl')
for row in results:
    print(row) #.concept, row.label_literal_en)


(rdflib.term.URIRef('http://www.w3.org/ns/dcat#Dataset'), rdflib.term.URIRef('http://purl.org/dc/terms/accessRights'), None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#Dataset'), rdflib.term.URIRef('http://www.w3.org/ns/dcat#contactPoint'), None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#Dataset'), rdflib.term.URIRef('http://purl.org/dc/terms/creator'), None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#Dataset'), rdflib.term.URIRef('http://purl.org/dc/terms/identifier'), None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#Dataset'), rdflib.term.URIRef('http://purl.org/dc/terms/publisher'), None)
(rdflib.term.URIRef('http://www.w3.org/ns/dcat#Dataset'), rdflib.term.URIRef('http://www.w3.org/ns/dcat#theme'), None)
